# 🏋️ 实验三：LoRA 微调训练（核心）

## 学习目标
- 配置 LoRA 适配器
- 使用 `trl` 的 `SFTTrainer` 进行指令微调
- 使用 `SwanLab` 实时监控训练过程
- 保存微调后的 LoRA 适配器权重

---

## 什么是 SFTTrainer？

`trl` 库的 `SFTTrainer` 是 HuggingFace Trainer 的封装，专门用于监督微调（Supervised Fine-Tuning），
内置了对话格式处理、数据拼接等功能，比直接用 Trainer 更方便。

## 1. 加载模型与分词器（NPU + FlashAttention 加速）

> **硬件说明**：本实验基于 **Ascend NPU** 运行。与 GPU 不同，Ascend 通过 `torch_npu` 驱动，
> 其 SDPA 算子会自动调用昇腾的硬件加速单元（达芬奇架构中的 Cube Unit）来加速注意力计算。

In [ ]:
import torch
import torch_npu  # noqa: F401 — 注册 Ascend NPU 后端
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

MODEL_PATH = "./model/deepseek-ai/deepseek-llm-7b-chat"

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{MODEL_PATH}/tokenizer.json",
    pad_token="</s>",
    eos_token="</s>",
    bos_token="<s>"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="npu:0",  # Ascend NPU
    attn_implementation="sdpa"  # 🔥 SDPA — 自动调用 NPU 的 FlashAttention 算子
)

print(f"模型已加载: {model.device}")

## 2. 配置 LoRA

使用 PEFT 库将 LoRA 适配器附加到模型上。

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

---

## 📖 拓展阅读：LoRA 微调原理详解

### 为什么需要 LoRA？

全参数微调（Full Fine-Tuning）一个 7B 模型需要：
- **显存**：约 4× 模型大小 ≈ **28GB**（仅优化器状态 + 梯度）+ 模型权重 14GB + 中间激活 ≈ **50-70GB 总显存**
- **时间**：全部 7B 参数都要计算梯度并更新

而 LoRA（Low-Rank Adaptation，Hu et al., 2021）通过 **冻结原始权重，只训练少量低秩矩阵**，大幅降低资源需求。

### LoRA 的核心思想

```
┌──────────────────────────────────────────────────┐
│                  全量微调                          │
│                                                    │
│   输入 x → [W (d×k) 可训练] → 输出 h               │
│                                                    │
│   ΔW 是 d×k 矩阵，参数量 = d×k                      │
│   对于 7B 模型，每个注意力投影矩阵约 0.5M 参数        │
└──────────────────────────────────────────────────┘
                      ↓
┌──────────────────────────────────────────────────┐
│                LoRA 低秩适配                       │
│                                                    │
│   输入 x → [W (d×k) 冻结] → 输出 h                 │
│              +                                     │
│            [B (d×r) · A (r×k)] 可训练               │
│                                                    │
│   参数量 = d×r + r×k = r×(d+k)                     │
│   r=8 时，仅为全量的 0.3%                           │
└──────────────────────────────────────────────────┘
```

**数学表达**：

$$h = W_0 x + \Delta W x = W_0 x + BAx$$

其中：
- $W_0 \in \mathbb{R}^{d \times k}$：冻结的原始权重矩阵
- $B \in \mathbb{R}^{d \times r}, A \in \mathbb{R}^{r \times k}$：可训练的低秩矩阵
- $r \ll \min(d, k)$：秩（rank），控制参数量

### 关键参数详解

**`r`（秩）—— 控制参数量**
- 低秩分解的维度，$r$ 越小，参数量越少
- $r=8$：每层仅增加约 0.3% 参数
- $r=64$：表达能力强，但参数量是 $r=8$ 的 8 倍
- **本实验取值 $r=8$**：在心理健康领域，领域知识较集中，低秩足以捕捉

**`lora_alpha`（缩放系数）—— 控制更新幅度**
- LoRA 权重缩放为 $\frac{\alpha}{r}$
- $\alpha=32$，$r=8$ → 缩放因子 = 4
- 较大的 $\alpha$ 放大 LoRA 的更新信号，适合新领域适配
- 较小的 $\alpha$ 更保守，适合与基座能力接近的任务

**`target_modules`（目标模块）—— 选择哪些层插入 LoRA**
- 本实验选择 `["q_proj", "v_proj"]`（Query 和 Value 投影矩阵）
- 为什么选 Q 和 V？
  - 研究表明 Q 和 V 对语义理解最重要
  - 只选 2 个模块可以控制参数量
  - 如果要更强的适配能力，可以加入 `k_proj`, `o_proj`, `up_proj`, `down_proj`

**`lora_dropout`（丢弃率）—— 防止过拟合**
- 在 LoRA 路径上随机丢弃一定比例的神经元
- 0.05 是常用值，小数据集上可以防止过拟合

### 本实验的 LoRA 配置

```python
LoraConfig(
    r=8,              # 低秩维度，参数量≈0.3%
    lora_alpha=32,    # 缩放系数，更新幅度适中
    target_modules=["q_proj", "v_proj"],  # 只适配注意力 Q 和 V
    lora_dropout=0.05,  # 轻量正则化
    bias="none",      # 不训练偏置项
    task_type="CAUSAL_LM"  # 因果语言模型
)
```

**可训练参数**：约 **4.2M**（总参数量 7B 的 **0.06%**），显存需求从 ~50GB 降至 ~16GB。

### LoRA + FlashAttention 协同工作

在本实验中，两个优化同时生效：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">优化</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">作用对象</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">效果</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>LoRA</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">参数更新方式</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">减少可训练参数 99.94%，降低显存和计算量</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>FlashAttention (SDPA)</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">注意力计算方式</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">减少注意力层的显存 $O(N^2) \to O(N)$，加速前向/反向传播</td>
    </tr>
  </tbody>
</table>

两者互补：
- LoRA 解决"参数量"的问题——只更新少量参数，降低优化器状态和梯度的显存
- FlashAttention 解决"计算量"的问题——通过 **Tiling + Online Softmax** 减少 HBM 访问，加速每次前向传播

> 💡 **在 Ascend NPU 上**：LoRA 的低秩矩阵在 NPU 的 AI Core 上高效执行，而 SDPA 算子利用 NPU 的 Cube Unit 加速矩阵乘法，两者各自发挥硬件优势。

### 参考文章

- [AIInfraGuide — FlashAttention V1 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/61-flashattention-v1详解/)
- [AIInfraGuide — FlashAttention V2 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/62-flashattention-v2详解/)

---

## 3. 加载数据集

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "src/medical_multi_data.json"},
    split="train"
)

def convert_conversation(conversation):
    text = ""
    for turn in conversation:
        if turn.get("system"):
            text += f"<|system|>\n{turn['system']}\n<|end|>\n"
        text += f"<|user|>\n{turn['input']}\n<|end|>\n"
        text += f"<|assistant|>\n{turn['output']}\n<|end|>\n"
    return text.strip()

def format_data(example):
    text = convert_conversation(example["conversation"])
    return {"text": text}

formatted_dataset = dataset.map(format_data, remove_columns=dataset.column_names)
split_data = formatted_dataset.train_test_split(test_size=0.1)
train_data = split_data["train"]
eval_data = split_data["test"]

print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(eval_data)} 条")

## 4. 配置 SwanLab

In [ ]:
import getpass
import swanlab
from pathlib import Path

%env SWANLAB_PROJ=DeepSeek-Medical-SFT
%env SWANLAB_EXP_NAME=DeepSeek-LoRA-Medical

def _swanlab_credential_saved() -> bool:
    netrc = Path.home() / ".netrc"
    if not netrc.exists():
        return False
    try:
        return "swanlab.cn" in netrc.read_text()
    except Exception:
        return False

if _swanlab_credential_saved():
    print("检测到本地已有 SwanLab 凭证，跳过登录。")
else:
    api_key = getpass.getpass("请输入你的 SwanLab API Key（从 https://swanlab.cn/settings 获取）: ")
    swanlab.login(api_key=api_key, save=True)
    print("SwanLab 登录成功！凭证已保存。")

## 5. 配置训练参数并开始训练

使用 `trl` 的 `SFTConfig` 和 `SFTTrainer`。

> ⏳ 训练约需 15-30 分钟。训练曲线可在 [swanlab.cn](https://swanlab.cn) 实时查看。

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=500,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    save_total_limit=2,
    bf16=True,
    report_to="swanlab",
    gradient_checkpointing=True,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_data,
    eval_dataset=eval_data,
)

trainer.train()
swanlab.finish()
print("训练完成！")

## 6. 保存模型

In [ ]:
model.save_pretrained("./sft_output", safe_serialization=True)
tokenizer.save_pretrained("./sft_output")
print(f"模型已保存到 ./sft_output")

import os
for f in os.listdir("./sft_output"):
    size = os.path.getsize(os.path.join("./sft_output", f))
    print(f"  {f:35s} {size/1024/1024:.2f} MB" if size > 1024*1024 else f"  {f:35s} {size/1024:.2f} KB")

## 小结

✅ 显式配置了 LoRA（$r=8$, `target_modules=["q_proj", "v_proj"]`）——可训练参数仅 **4.2M（0.06%）**
✅ 理解了 LoRA 的低秩分解原理：$h = W_0x + BAx$，用 $B$ 和 $A$ 低秩矩阵模拟权重更新
✅ 使用 `SFTTrainer` 完成了指令微调
✅ 启用了 **SDPA（FlashAttention）算子加速**——在 Ascend NPU 上通过 `torch_npu` 调用硬件加速的注意力计算
✅ LoRA + FlashAttention 协同工作，兼顾参数量与计算效率
✅ 使用 SwanLab 实时监控了训练过程
✅ 保存了微调后的模型权重（LoRA 适配器，约 15MB）

下一节将测试微调后的模型效果。

## 课后练习

1. (单选题) 线性层权重 W0 形状 [4096,4096]，LoRA r=8 时该层新增参数量为？
   - A. 65,536
   - B. 4,096
   - C. 32,768
   - D. 16,777,216

2. (单选题) per_device_train_batch_size=1、gradient_accumulation_steps=8、2 卡并行时等效 batch 为？
   - A. 16
   - B. 8
   - C. 1
   - D. 64

3. (多选题) LoRA 降低训练显存的原因包括？
   - A. 冻结原始权重，不保存其优化器状态
   - B. 仅低秩矩阵参与梯度计算
   - C. 减少需要保存的梯度数量
   - D. 激活值完全不需要保存

4. (多选题) 影响 LoRA 表达能力的配置包括？
   - A. r
   - B. alpha
   - C. target_modules
   - D. lora_dropout

5. (判断题) LoRA 的缩放系数为 alpha/r，改变 r 时需同步考虑 alpha。

6. (判断题) r 越大一定带来更高准确率。

7. (填空题) 本实验 q_proj/v_proj 上 LoRA 可训练参数约为 ____，占总参数比例约 ____。

8. (填空题) SFTTrainer 中 max_seq_length 决定训练样本截断/填充后的 ____。

9. (简答题) 为什么只选择 q_proj 与 v_proj 而不是全部线性层？

10. (简答题) bf16 与 LoRA 如何协同降低单卡训练 7B 模型的显存压力？

11. (代码设计题) 写出 PEFT LoRA 配置与 model.print_trainable_parameters() 检查片段。

12. (单选题) 训练 loss 下降但验证 loss 上升，首先应？
   - A. 降低 r 或增大 dropout，并回滚到最佳 checkpoint
   - B. 增大 r
   - C. 提高学习率
   - D. 移除 LoRA

13. (多选题) SwanLab 可用于？
   - A. 绘制 loss/lr 曲线
   - B. 查看梯度范数
   - C. 多组实验对比
   - D. 替代模型保存

14. (判断题) 同时设置 max_steps 与 num_train_epochs 时，max_steps 优先控制训练步数。

15. (简答题) 对比 r=8 与 r=32 在参数量、显存、过拟合风险和收敛速度上的差异。

> 参考答案见 answer/05.04_lora_training_answer.ipynb。